In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

In [ ]:
train = pd.read_csv(r"projects/retail_analytics/data/raw/rossmann/train_v2.csv")
test = pd.read_csv(r"projects/retail_analytics/data/raw/rossmann/test_v2.csv")
store_df = pd.read_csv(r"projects/retail_analytics/data/raw/rossmann/store.csv")

# Keep the datasets separate, only merge store info into each one
for df in [train, test]:
    if "Store" in df.columns:
        df["Store"] = df["Store"].astype(str)

store_df["Store"] = store_df["Store"].astype(str)
store_df = store_df.drop_duplicates(subset="Store")

train_merged = train.merge(store_df, how="left", on="Store")
test_merged = test.merge(store_df, how="left", on="Store")

train_merged.head()
test_merged.head()

In [ ]:
train_merged["Date"] = pd.to_datetime(train_merged["Date"])
test_merged["Date"] = pd.to_datetime(test_merged["Date"])

train_merged = train_merged[train_merged["Open"] == 1].copy()
test_merged = test_merged[test_merged["Open"] == 1].copy()

In [ ]:
train_merged.columns

In [ ]:
plt.bar(train_merged.DayOfWeek,train_merged.Sales)

In [ ]:
plt.bar(train_merged.Promo,train_merged.Sales)

In [ ]:
# train_merged.corr()["Sales"].sort_values(ascending=False)
train_merged["Day"] = train_merged.Date.dt.day
train_merged["Month"] = train_merged.Date.dt.month
train_merged["Year"] = train_merged.Date.dt.year

test_merged["Day"] = test_merged.Date.dt.day
test_merged["Month"] = test_merged.Date.dt.month
test_merged["Year"] = test_merged.Date.dt.year

In [ ]:
yearlySales = train_merged.groupby("Year")["Sales"].mean()
plt.bar(yearlySales.index, yearlySales.values)
plt.xlabel("Year")
plt.ylabel("Average Sales")
plt.title("Average Sales by Year")
plt.show()

In [ ]:
train_merged.columns

In [ ]:
input_cols = ['Store', 'DayOfWeek', 'Promo', 'StateHoliday', 'StoreType', 'Assortment', 'Day', 'Month', 'Year']
target_col = 'Sales'

train_model = train_merged[input_cols + [target_col]].copy()
test_features = test_merged[input_cols].copy()

for col in ['Store', 'StateHoliday', 'StoreType', 'Assortment']:
    train_model[col] = train_model[col].astype(str)
    test_features[col] = test_features[col].astype(str)

In [ ]:
X_train = train_model[input_cols].copy()
X_test = test_features[input_cols].copy()
y_train = train_model[target_col].copy()

numeric_cols = [col for col in input_cols if X_train[col].dtype.kind in 'iuf']
categorical_cols = [col for col in input_cols if col not in numeric_cols]

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
X_train[numeric_cols] = imputer.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = imputer.transform(X_test[numeric_cols])

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [ ]:
from sklearn.preprocessing import OneHotEncoder

if categorical_cols:
    encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    X_train_cat = encoder.fit_transform(X_train[categorical_cols])
    X_test_cat = encoder.transform(X_test[categorical_cols])
    cat_names = encoder.get_feature_names_out(categorical_cols)

    X_train = pd.concat(
        [X_train.drop(columns=categorical_cols).reset_index(drop=True),
         pd.DataFrame(X_train_cat, columns=cat_names, index=X_train.index)],
        axis=1
    )
    X_test = pd.concat(
        [X_test.drop(columns=categorical_cols).reset_index(drop=True),
         pd.DataFrame(X_test_cat, columns=cat_names, index=X_test.index)],
        axis=1
    )

X_train.head()
X_test.head()